# Food Object Detection with YOLOv8

**Pipeline overview**
1. Mount Google Drive & extract raw data
2. Visualise bounding boxes
3. Pre-process dataset (dedup, blur detection, resize, augmentation)
4. Split into train / test
5. Train YOLOv8n (baseline) and YOLOv8m (tuned)
6. Evaluate both models and compare performance

> **Note:** Designed for Google Colab with a Wajahat_Project folder in Google Drive.
> Update the path constants in each section if your folder structure differs.

## 0 · Install dependencies

In [ ]:
!pip install ultralytics rarfile tqdm opencv-python-headless --quiet

## 1 · Mount Drive & Extract Data

In [ ]:
import sys, os
sys.path.insert(0, '/content')  # allow importing local src modules

from google.colab import drive
drive.mount('/content/drive')

import rarfile

BASE      = '/content/drive/MyDrive/Wajahat_Project'
RAW_ZIP   = f'{BASE}/01_Raw_Dataset/Raw_Dataset.rar'
ANN_ZIP   = f'{BASE}/02_Annotations/Annotations.rar'
RAW_OUT   = '/content/raw_dataset'
ANN_OUT   = '/content/sample_data'

os.makedirs(RAW_OUT, exist_ok=True)
os.makedirs(ANN_OUT, exist_ok=True)

for archive, dest in [(RAW_ZIP, RAW_OUT), (ANN_ZIP, ANN_OUT)]:
    try:
        with rarfile.RarFile(archive, 'r') as z:
            z.extractall(dest)
        print(f'Extracted {archive} → {dest}')
    except rarfile.BadRarFile as e:
        print(f'ERROR: {e}')

RAW_IMG_DIR = os.path.join(RAW_OUT, 'Raw_Dataset')
ANN_DIR     = os.path.join(ANN_OUT, 'Annotations')

imgs = [f for f in os.listdir(RAW_IMG_DIR) if f.lower().endswith(('.jpg','.jpeg','.png'))]
anns = [f for f in os.listdir(ANN_DIR) if f.endswith('.txt')]
print(f'Images: {len(imgs)}  |  Annotations: {len(anns)}')

## 2 · Visualise Bounding Boxes

In [ ]:
import cv2
import matplotlib.pyplot as plt

def load_yolo_annotation(txt_path):
    boxes = []
    with open(txt_path, 'r') as f:
        for line in f:
            parts = line.strip().split()
            cls, cx, cy, bw, bh = int(parts[0]), *map(float, parts[1:])
            boxes.append((cls, cx, cy, bw, bh))
    return boxes

def draw_boxes(img_bgr, boxes):
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    H, W, _ = img_rgb.shape
    for cls, cx, cy, bw, bh in boxes:
        x1 = int((cx - bw/2)*W); y1 = int((cy - bh/2)*H)
        x2 = int((cx + bw/2)*W); y2 = int((cy + bh/2)*H)
        cv2.rectangle(img_rgb, (x1,y1),(x2,y2),(255,0,0),2)
        cv2.putText(img_rgb, str(cls),(x1,max(y1-5,0)),
                    cv2.FONT_HERSHEY_SIMPLEX,0.7,(255,0,0),2)
    return img_rgb

sample = imgs[4]
img    = cv2.imread(os.path.join(RAW_IMG_DIR, sample))
ann    = load_yolo_annotation(os.path.join(ANN_DIR, os.path.splitext(sample)[0]+'.txt'))
vis    = draw_boxes(img, ann)

plt.figure(figsize=(8,8))
plt.imshow(vis); plt.axis('off'); plt.title(sample); plt.show()

## 3 · Pre-processing

In [ ]:
import hashlib, shutil
import numpy as np
from tqdm import tqdm

OUT_DIR = f'{BASE}/Preprocessed_dataset'
AUG_DIR = os.path.join(OUT_DIR, 'augmented')
IMG_SIZE = 640

os.makedirs(OUT_DIR, exist_ok=True)

# ── 3a Remove duplicates ──────────────────────────────────────────────
hashes, duplicates = {}, []
for img in tqdm(imgs, desc='Dedup'):
    path = os.path.join(RAW_IMG_DIR, img)
    h = hashlib.md5(open(path,'rb').read()).hexdigest()
    if h in hashes: duplicates.append(img)
    else:           hashes[h] = img
for d in duplicates: os.remove(os.path.join(RAW_IMG_DIR, d))
print(f'Duplicates removed: {len(duplicates)}')

# ── 3b Blur detection ─────────────────────────────────────────────────
bad = []
for img in tqdm(imgs, desc='Blur'):
    path = os.path.join(RAW_IMG_DIR, img)
    if not os.path.exists(path): continue
    var = cv2.Laplacian(cv2.imread(path, cv2.IMREAD_GRAYSCALE), cv2.CV_64F).var()
    if var < 60: bad.append(img)
print(f'Blurry images: {len(bad)}')

# ── 3c Resize & normalise ─────────────────────────────────────────────
valid_imgs = [f for f in os.listdir(RAW_IMG_DIR) if f.lower().endswith('.jpg')]
for img_name in tqdm(valid_imgs, desc='Resize'):
    img_bgr = cv2.imread(os.path.join(RAW_IMG_DIR, img_name))
    if img_bgr is None: continue
    resized = cv2.resize(img_bgr, (IMG_SIZE, IMG_SIZE))
    cv2.imwrite(os.path.join(OUT_DIR, img_name), resized)
    base = os.path.splitext(img_name)[0]
    src  = os.path.join(ANN_DIR, f'{base}.txt')
    if os.path.exists(src):
        shutil.copy(src, os.path.join(OUT_DIR, f'{base}.txt'))
print('Resize complete.')

In [ ]:
# ── 3d Augmentation ───────────────────────────────────────────────────
os.makedirs(AUG_DIR, exist_ok=True)

def augment_image(img_path, ann_path, img_name, aug_dir):
    img = cv2.imread(img_path)
    H, W = img.shape[:2]
    lines = open(ann_path).readlines()
    base  = os.path.splitext(img_name)[0]

    for angle in (-15, 15):
        M  = cv2.getRotationMatrix2D((W//2, H//2), angle, 1.0)
        rt = cv2.warpAffine(img, M, (W,H))
        cv2.imwrite(f'{aug_dir}/{base}_rot{angle:+d}.jpg', rt)
        open(f'{aug_dir}/{base}_rot{angle:+d}.txt','w').writelines(lines)

    flipped = cv2.flip(img, 1)
    cv2.imwrite(f'{aug_dir}/{base}_flip.jpg', flipped)
    flip_lines = []
    for ln in lines:
        p = ln.strip().split()
        flip_lines.append(f'{p[0]} {1-float(p[1]):.6f} {p[2]} {p[3]} {p[4]}\n')
    open(f'{aug_dir}/{base}_flip.txt','w').writelines(flip_lines)

    for delta in (40, -40):
        bright = np.clip(img.astype(np.int16)+delta,0,255).astype(np.uint8)
        cv2.imwrite(f'{aug_dir}/{base}_b{delta:+d}.jpg', bright)
        open(f'{aug_dir}/{base}_b{delta:+d}.txt','w').writelines(lines)

proc_imgs = [f for f in os.listdir(OUT_DIR) if f.endswith('.jpg')]
for img_name in tqdm(proc_imgs, desc='Augmenting'):
    ip = os.path.join(OUT_DIR, img_name)
    ap = os.path.join(OUT_DIR, img_name.replace('.jpg','.txt'))
    if os.path.exists(ap):
        augment_image(ip, ap, img_name, AUG_DIR)
print('Augmentation done.')

## 4 · Train / Test Split

In [ ]:
import random

SPLIT_DIR     = f'{BASE}/Dataset_Split'
TRAIN_IMG_DIR = os.path.join(SPLIT_DIR, 'train/images')
TRAIN_LBL_DIR = os.path.join(SPLIT_DIR, 'train/labels')
TEST_IMG_DIR  = os.path.join(SPLIT_DIR, 'test/images')
TEST_LBL_DIR  = os.path.join(SPLIT_DIR, 'test/labels')

for d in (TRAIN_IMG_DIR, TRAIN_LBL_DIR, TEST_IMG_DIR, TEST_LBL_DIR):
    os.makedirs(d, exist_ok=True)

all_imgs = [f for f in os.listdir(OUT_DIR) if f.endswith('.jpg')]
random.seed(42)
random.shuffle(all_imgs)
split = int(len(all_imgs) * 0.8)
train_images = all_imgs[:split]
test_images  = all_imgs[split:]

def copy_pairs(img_list, img_dest, lbl_dest):
    for nm in tqdm(img_list, desc=f'→ {os.path.basename(img_dest)}'):
        shutil.copy(os.path.join(OUT_DIR, nm), img_dest)
        lbl = os.path.join(OUT_DIR, nm.replace('.jpg','.txt'))
        if os.path.exists(lbl):
            shutil.copy(lbl, lbl_dest)

copy_pairs(train_images, TRAIN_IMG_DIR, TRAIN_LBL_DIR)
copy_pairs(test_images,  TEST_IMG_DIR,  TEST_LBL_DIR)
print(f'Train: {len(train_images)}  |  Test: {len(test_images)}')

In [ ]:
# Preview training samples
plt.figure(figsize=(10,10))
for i, nm in enumerate(train_images[:9]):
    img = cv2.cvtColor(cv2.imread(os.path.join(TRAIN_IMG_DIR, nm)), cv2.COLOR_BGR2RGB)
    plt.subplot(3,3,i+1); plt.imshow(img); plt.axis('off')
plt.suptitle('Sample Augmented Training Images', fontsize=16)
plt.tight_layout(); plt.show()

## 5 · Create food.yaml

In [ ]:
yaml_content = f"""\
path: {SPLIT_DIR}
train: train/images
val: test/images
nc: 10
names: ['Breads','Curry','Pizza','Rice','Roti_Chappati','Cake','Drink','Egg','Fruit','Salad']
"""
yaml_path = f'{BASE}/food.yaml'
os.makedirs(os.path.dirname(yaml_path), exist_ok=True)
open(yaml_path,'w').write(yaml_content)
print(f'Created: {yaml_path}\n{yaml_content}')

## 6 · Train YOLOv8n (Baseline)

In [ ]:
!pip install ultralytics --quiet
from ultralytics import YOLO

model_n = YOLO('yolov8n.pt')
model_n.train(
    data=yaml_path,
    epochs=50,
    imgsz=640,
    batch=16,
    name='food_detection_yolov8n'
)

## 7 · Train YOLOv8m (Tuned)

In [ ]:
model_m = YOLO('yolov8m.pt')
model_m.train(
    data=yaml_path,
    epochs=150,
    imgsz=640,
    batch=8,
    name='food_detection_yolov8m_tuned'
)

## 8 · Evaluate Both Models

In [ ]:
# Load best weights
nano_path   = '/content/runs/detect/food_detection_yolov8n/weights/best.pt'
medium_path = '/content/runs/detect/food_detection_yolov8m_tuned2/weights/best.pt'

model_nano   = YOLO(nano_path)
model_medium = YOLO(medium_path)

metrics_n = model_nano.val(data=yaml_path)
metrics_m = model_medium.val(data=yaml_path)

print('\n── YOLOv8n ──')
print(f'mAP50: {metrics_n.box.map50:.4f}  |  mAP50-95: {metrics_n.box.map:.4f}')

print('\n── YOLOv8m ──')
print(f'mAP50: {metrics_m.box.map50:.4f}  |  mAP50-95: {metrics_m.box.map:.4f}')

print('\n── Class-wise AP50 comparison ──')
print(f'{"Class":<18} {"YOLOv8n":>12} {"YOLOv8m":>12}')
for name, n, m in zip(metrics_n.names.values(), metrics_n.box.ap, metrics_m.box.ap):
    print(f'{name:<18} {n:>12.4f} {m:>12.4f}')

## 9 · Display Evaluation Plots

In [ ]:
def show_plots(val_dir, label=''):
    plots = {
        'F1 Curve'        : 'BoxF1_curve.png',
        'Precision Curve' : 'BoxP_curve.png',
        'Recall Curve'    : 'BoxR_curve.png',
        'Confusion Matrix': 'confusion_matrix.png',
    }
    for title, fname in plots.items():
        path = os.path.join(val_dir, fname)
        if not os.path.exists(path):
            print(f'[WARN] Not found: {path}'); continue
        img = cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB)
        plt.figure(figsize=(10,7))
        plt.imshow(img); plt.axis('off')
        plt.title(f'{label} – {title}' if label else title)
        plt.show()

show_plots('/content/runs/detect/val',  'YOLOv8n')
show_plots('/content/runs/detect/val2', 'YOLOv8m (tuned)')